# ???????? YOLOv8 ??? ???????? ???????? ?? ???

??????? ??? ??????? ? Google Colab. ????????? ??????? ? Kaggle, ??????? YOLOv8 ? ????????? ???? ? ??????? ? Google Drive.


In [ ]:
!pip install -q ultralytics kagglehub PyYAML

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_NAME = 'brain-mri-yolov8'
WORKDIR = Path('/content') / PROJECT_NAME
ARTIFACTS_DIR = Path('/content/drive/MyDrive') / PROJECT_NAME
DATASET_DIR = WORKDIR / 'dataset'
YOLO_DIR = WORKDIR / 'yolo_dataset'
RUNS_DIR = ARTIFACTS_DIR / 'runs'

MODEL_NAME = 'yolov8n.pt'
EPOCHS = 30
IMAGE_SIZE = 512
BATCH_SIZE = 16
SEED = 42

CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
SPLIT_MAPPING = {
    'Train': 'train',
    'Val': 'val',
}

WORKDIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
YOLO_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('WORKDIR:', WORKDIR)
print('ARTIFACTS_DIR:', ARTIFACTS_DIR)

In [ ]:
import kagglehub
import shutil
from pathlib import Path

path = kagglehub.dataset_download('ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes')
dataset_path = Path(path)
print('Raw kagglehub path:', dataset_path)

candidate_roots = [dataset_path] + [p for p in dataset_path.rglob('*') if p.is_dir()]
resolved_root = None
for candidate in candidate_roots:
    if (candidate / 'Train').exists() and (candidate / 'Val').exists():
        resolved_root = candidate
        break

if resolved_root is None:
    raise FileNotFoundError('Не удалось найти папки Train/Val внутри датасета, скачанного через kagglehub.')

dataset_target = DATASET_DIR / 'brain-tumor-mri'
if dataset_target.exists():
    shutil.rmtree(dataset_target)
shutil.copytree(resolved_root, dataset_target)

print('Resolved dataset root:', resolved_root)
print('Copied dataset to:', dataset_target)

In [ ]:
import shutil
from collections import defaultdict

SOURCE_ROOT = DATASET_DIR / 'brain-tumor-mri'

if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
YOLO_DIR.mkdir(parents=True, exist_ok=True)

summary = defaultdict(lambda: {'images': 0, 'labels': 0, 'boxes': 0})
missing_labels = 0
empty_labels = 0

for src_split, dst_split in SPLIT_MAPPING.items():
    images_out = YOLO_DIR / dst_split / 'images'
    labels_out = YOLO_DIR / dst_split / 'labels'
    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    split_root = SOURCE_ROOT / src_split
    for class_dir in sorted([p for p in split_root.iterdir() if p.is_dir()]):
        class_name = class_dir.name
        for image_path in sorted((class_dir / 'images').glob('*')):
            target_stem = f"{class_name.lower().replace(' ', '_')}__{image_path.stem}"
            image_target = images_out / f"{target_stem}{image_path.suffix.lower()}"
            shutil.copy2(image_path, image_target)
            summary[dst_split]['images'] += 1

            label_path = class_dir / 'labels' / f"{image_path.stem}.txt"
            label_target = labels_out / f"{target_stem}.txt"
            if label_path.exists():
                content = label_path.read_text(encoding='utf-8').strip()
                label_target.write_text(content, encoding='utf-8')
                summary[dst_split]['labels'] += 1
                if content:
                    summary[dst_split]['boxes'] += len(content.splitlines())
                else:
                    empty_labels += 1
            else:
                label_target.write_text('', encoding='utf-8')
                missing_labels += 1

print('Dataset prepared for YOLO at:', YOLO_DIR)
print('Summary:', dict(summary))
print('Missing labels:', missing_labels)
print('Empty labels:', empty_labels)

In [ ]:
import yaml

dataset_yaml = {
    'path': str(YOLO_DIR),
    'train': str((YOLO_DIR / 'train' / 'images').resolve()),
    'val': str((YOLO_DIR / 'val' / 'images').resolve()),
    'names': CLASS_NAMES,
}

yaml_path = WORKDIR / 'brain_mri_yolo.yaml'
with yaml_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(dataset_yaml, handle, sort_keys=False, allow_unicode=True)

print(yaml_path.read_text(encoding='utf-8'))

In [ ]:
from collections import Counter
import json

class_counter = {split: Counter() for split in SPLIT_MAPPING.values()}
for src_split, dst_split in SPLIT_MAPPING.items():
    split_root = SOURCE_ROOT / src_split
    for class_dir in sorted([p for p in split_root.iterdir() if p.is_dir()]):
        image_count = len(list((class_dir / 'images').glob('*')))
        class_counter[dst_split][class_dir.name] = image_count

dataset_summary = {
    'class_names': CLASS_NAMES,
    'splits': {split: dict(counter) for split, counter in class_counter.items()},
    'prepared_summary': dict(summary),
    'missing_labels': missing_labels,
    'empty_labels': empty_labels,
}

summary_path = ARTIFACTS_DIR / 'dataset_summary.json'
summary_path.write_text(json.dumps(dataset_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(dataset_summary, ensure_ascii=False, indent=2))

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

def show_sample_grid(images_dir, labels_dir, num_samples=6):
    image_paths = sorted(images_dir.glob('*'))
    selected = random.sample(image_paths, k=min(num_samples, len(image_paths)))
    plt.figure(figsize=(14, 10))

    for idx, image_path in enumerate(selected, start=1):
        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        label_path = labels_dir / f'{image_path.stem}.txt'
        if label_path.exists():
            for line in label_path.read_text(encoding='utf-8').strip().splitlines():
                if not line.strip():
                    continue
                class_id, x_center, y_center, box_width, box_height = map(float, line.split())
                x1 = int((x_center - box_width / 2) * w)
                y1 = int((y_center - box_height / 2) * h)
                x2 = int((x_center + box_width / 2) * w)
                y2 = int((y_center + box_height / 2) * h)
                class_name = CLASS_NAMES[int(class_id)]
                cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(image, class_name, (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        plt.subplot(2, 3, idx)
        plt.imshow(image)
        plt.axis('off')
        plt.title(image_path.name)

    plt.tight_layout()
    plt.show()

show_sample_grid(YOLO_DIR / 'train' / 'images', YOLO_DIR / 'train' / 'labels')

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=str(RUNS_DIR),
    name='yolov8n_brain_mri',
    seed=SEED,
    pretrained=True,
    exist_ok=True,
    verbose=True,
)

train_results

In [ ]:
metrics = model.val(data=str(yaml_path), split='val')
metrics

In [ ]:
from pathlib import Path
import shutil

run_dir = RUNS_DIR / 'yolov8n_brain_mri'
weights_dir = run_dir / 'weights'
best_weights = weights_dir / 'best.pt'
last_weights = weights_dir / 'last.pt'

final_dir = ARTIFACTS_DIR / 'final_artifacts'
final_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    best_weights,
    last_weights,
    run_dir / 'results.csv',
    run_dir / 'results.png',
    run_dir / 'confusion_matrix.png',
    run_dir / 'confusion_matrix_normalized.png',
]

for file_path in files_to_copy:
    if file_path.exists():
        shutil.copy2(file_path, final_dir / file_path.name)

print('Artifacts directory:', final_dir)
print('Best weights:', best_weights)
print('Last weights:', last_weights)
print('Available files:')
for path in sorted(final_dir.glob('*')):
    print('-', path.name)

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image

best_model = YOLO(str(best_weights))
sample_images = sorted((YOLO_DIR / 'val' / 'images').glob('*'))[:6]
prediction_dir = ARTIFACTS_DIR / 'predictions'
prediction_dir.mkdir(parents=True, exist_ok=True)

results = best_model.predict([str(path) for path in sample_images], conf=0.25, save=True, project=str(prediction_dir), name='val_preview', exist_ok=True)

saved_images = sorted((prediction_dir / 'val_preview').glob('*'))
plt.figure(figsize=(14, 10))
for idx, image_path in enumerate(saved_images[:6], start=1):
    image = Image.open(image_path)
    plt.subplot(2, 3, idx)
    plt.imshow(image)
    plt.axis('off')
    plt.title(image_path.name)
plt.tight_layout()
plt.show()

## ??? ????????? ????? ????????

????? ??????? ????? ??????? ?? Google Drive `best.pt`, `last.pt`, `results.csv`, `results.png` ? `dataset_summary.json`. ??? ????? ????? ????? ????????? ? ????????? ???????????.
